# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadar2846/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, subprocess
import pandas as pd, numpy as np

REPO_URL = "https://github.com/saadar2846/flyrank_ml_internship"
REPO_DIR = "flyrank_ml_internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(os.getcwd())
print(os.listdir())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

/content/flyrank_ml_internship/flyrank_ml_internship
['SETUP.md', 'LICENSE', 'README.md', 'DATA_USE.md', 'work', '.git', 'submission', 'requirements.txt', '.gitignore', 'scripts', 'docs', 'GUIDE.md', 'data', 'skills', 'notebooks', 'CLAUDE.md', 'outputs', '02_your_first_readable_model.ipynb', 'AGENTS.md', '.github']
(30000, 45)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Classification**

Lane 1 is fundamentally about testing which signals are associated with outcomes — that's not
naturally "ranking" (Lane 2/4's job) or "clustering" (Lane 3's job). To test signal strength
rigorously rather than just eyeballing correlations, I'll frame it as binary classification: predict
whether a page is declining, using observable signals as features, then read the model's feature
importances and individual correlations as the actual signal-audit output. The classification itself
is a means to an end — the real deliverable is "which signals carry real information," not the
prediction itself.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy):** `is_declining_label = (trend_direction == "down")` — reused from the starter
pipeline. This is explicitly a proxy, not a true outcome: it's a bucket calculated from the *current*
window, not a future observed decline. I'm using it here because Week 1/2 already validated it works
end-to-end, and Lane 1's job this week is auditing signals against *a* target, not perfecting the
target itself .

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Average Precision (AP)**, with per-signal correlation as a companion number.

Accuracy is misleading here because declining vs. non-declining pages aren't balanced. AP reflects
whether the signals can separate the two classes across all thresholds, not just at one cutoff. "Good"
means AP that clearly beats a naive baseline (the starter's baseline rule scored 0.468 AP; anything
meaningfully above that, on a proper holdout, would defend this as a real signal, not noise).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one content page** (`content_id`), described by its observed 90-day
signals. Real slice shown below.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "content_type"]

unit_view = df[["content_id"] + features + ["is_declining_label"]].head(10)
unit_view

,content_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,content_type,is_declining_label
0,content_304f48230142,187,20,3803,10.6,0.76,3221.0,keyword article,1
1,content_a1fb4e703a9e,445,25,15320,20.3,0.05,2481.0,keyword article,1
2,content_9aa793d4d895,141,20,12581,36.5,0.09,3515.0,keyword article,1
3,content_331d6c4de07b,463,22,11751,6.2,0.49,NaN,keyword article,0
4,content_d99b7a2d90ca,263,14,19140,44.0,0.13,2803.0,keyword article,1
5,content_d4084a4bc775,147,20,3970,8.5,0.03,3080.0,keyword article,1
6,content_9a34b442b552,90,20,20,7.0,0.00,3059.0,keyword article,1
7,content_a63219c6e95a,445,22,1724,21.2,0.06,NaN,keyword article,0
8,content_5e6c160719bc,90,20,32574,46.0,0.09,3807.0,keyword article,1
9,content_c27558df2b0c,257,104,1240,4.9,0.16,NaN,keyword article,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why this is too messy for an if-statement:** signals interact, they don't just add up. A page's age
only predicts decline when it's paired with low impressions — an old page with strong, steady demand
isn't actually at risk. A single if-statement threshold (e.g. "age > 180 days") can't express that
conditional relationship; it either flags too many stable old pages or misses young pages that are
already declining for other reasons. This isn't hypothetical: the starter pipeline's hand-written rule
(stale AND visible) hit Precision@50 of 0.240, while a random forest — able to learn these interactions
— hit 0.740 on the same data. That 3x gap is the evidence that the pattern is genuinely too tangled
for a fixed rule.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.